# Feature Visualization

This notebook loads:
1. Trained probe weights and configurations from WandB
2. Activations from HuggingFace  
3. Questions, labels, and model outputs from the dataset files

In [23]:
%%capture
# If getting 'Could not find project LASR_probe_gen' get key from https://wandb.ai/authorize and paste below
import os
os.environ["WANDB_SILENT"] = "true"
import wandb
WANDB_KEY = os.getenv("WANDB_KEY")
wandb.login(key=WANDB_KEY)

In [24]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import torch
import tqdm

import probe_gen.probes as probes
from probe_gen.config import ConfigDict, data
from probe_gen.standard_experiments.hyperparameter_search import get_best_hyperparams_for_train_setup


## Configuration

Set the parameters for which probe and activations to load


In [25]:
# Configuration parameters (same as in TrainProbe.ipynb)
probe_type = ["mean", "attention_torch", "mean_torch"][1]
behaviour = "science"
datasource = "mmlu"
activations_model = "llama_3b"
generation_method = "on_policy"
response_model = "llama_3b"
off_policy_model = "qwen_3b"
mode = "train"

print(f"Configuration:")
print(f"  Probe type: {probe_type}")
print(f"  Behaviour: {behaviour}")
print(f"  Datasource: {datasource}")
print(f"  Model: {activations_model}")
print(f"  Generation: {generation_method}")
print(f"  Mode: {mode}")

Configuration:
  Probe type: attention_torch
  Behaviour: science
  Datasource: mmlu
  Model: llama_3b
  Generation: on_policy
  Mode: train


## 1. Get Best Hyperparameters from WandB

Load the best hyperparameters for this probe configuration from WandB or local cache


In [26]:
# Load best hyperparameters from WandB or local cache
used_model = response_model if generation_method != "off_policy" else off_policy_model

try:
    # Try loading from local cache first
    cfg = ConfigDict.from_json(activations_model, probe_type, behaviour)
    print(f"✓ Loaded hyperparameters from local cache")
except KeyError:
    # Fall back to loading from WandB
    print(f"Loading hyperparameters from WandB...")
    train_setup = [[probe_type, behaviour, datasource, activations_model, 
                    generation_method, used_model, mode]]
    train_setup = get_best_hyperparams_for_train_setup(train_setup)
    cfg = train_setup[0][7]
    print(f"✓ Loaded hyperparameters from WandB")

print(f"\nBest hyperparameters:")
print(f"  Layer: {cfg.layer}")
print(f"  Use bias: {cfg.use_bias}")
print(f"  Normalize: {cfg.normalize}")
if "torch" in probe_type:
    print(f"  Learning rate: {cfg.lr}")
    print(f"  Weight decay: {cfg.weight_decay}")
else:
    print(f"  C (regularization): {cfg.C}")

✓ Loaded hyperparameters from local cache

Best hyperparameters:
  Layer: 12
  Use bias: True
  Normalize: True
  Learning rate: 0.0001
  Weight decay: 1e-05


## 2. Load Activations from HuggingFace

Load the pre-computed activations at the specified layer


In [27]:
# Load activations and labels from HuggingFace
used_model = response_model if generation_method != "off_policy" else off_policy_model
activations_tensor, attention_mask, labels_tensor = probes.load_hf_activations_at_layer(
    behaviour, 
    datasource, 
    activations_model, 
    used_model, 
    generation_method, 
    mode, 
    cfg.layer, 
    and_labels=True, 
    verbose=True
)

print(f"\nLoaded data:")
print(f"  Activations shape: {activations_tensor.shape}")
print(f"  Attention mask shape: {attention_mask.shape}")
print(f"  Labels shape: {labels_tensor.shape}")
print(f"  Positive samples: {labels_tensor.sum().item():.0f}")
print(f"  Negative samples: {(len(labels_tensor) - labels_tensor.sum()).item():.0f}")


KeyboardInterrupt: 

## 3. Load Questions and Model Outputs

Load the original questions and model responses from the JSONL dataset files


In [ ]:
# Load the JSONL file that contains questions, model outputs, and labels
used_model = response_model if generation_method != "off_policy" else off_policy_model
generation_method_for_labels = generation_method.replace("_included", "")
labels_filepath = f"{datasource}/{used_model}_{generation_method_for_labels}_{mode}.jsonl"
labels_localpath = data / behaviour / labels_filepath

print(f"Loading dataset from: {labels_localpath}")

# Load the JSONL file
data_rows = []
with open(labels_localpath, 'r') as file:
    for line in file:
        data_dict = json.loads(line)
        data_rows.append(data_dict)

# Convert to DataFrame for easier manipulation
dataset_df = pd.DataFrame(data_rows)

print(f"\n✓ Loaded {len(dataset_df)} samples from dataset")
print(f"\nAvailable columns: {list(dataset_df.columns)}")
print(f"\nFirst sample:")
if 'input' in dataset_df.columns:
    print(f"  Input: {dataset_df.iloc[0]['input'][:200]}...")
if 'model_outputs' in dataset_df.columns:
    print(f"  Output: {dataset_df.iloc[0]['model_outputs'][:200]}...")
if 'scale_labels' in dataset_df.columns:
    print(f"  Label: {dataset_df.iloc[0]['scale_labels']}")


## 4. Train Probe with Best Hyperparameters

Train the probe to get the weights for feature visualization


In [ ]:
# Aggregate activations if using mean probe
if "mean" in probe_type:
    activations_aggregated = probes.MeanAggregation()(activations_tensor, attention_mask)
    print(f"Aggregated activations shape: {activations_aggregated.shape}")
else:
    activations_aggregated = activations_tensor
    print(f"Using full sequence activations: {activations_aggregated.shape}")

# Create train and validation datasets
split_val = 2500 if behaviour in ["deception", "sandbagging"] else 3500
split_val = 3000 if datasource == "shakespeare" else split_val

train_dataset, val_dataset, test_dataset = probes.create_activation_datasets(
    activations_aggregated, labels_tensor, splits=[split_val, 500, 0], verbose=True)

# Initialize probe
if probe_type == "mean":
    probe = probes.SklearnLogisticProbe(cfg)
elif probe_type == "mean_torch":
    probe = probes.TorchLinearProbe(cfg)
elif probe_type == "attention_torch":
    probe = probes.TorchAttentionProbe(cfg)

print(f"\n✓ Created {probe_type} probe")

# Train the probe
print(f"\nTraining probe...")
probe.fit(train_dataset, val_dataset)

# Evaluate on validation set
eval_dict, y_pred, y_pred_proba = probe.eval(val_dataset)
print(f'\n✓ Validation ROC-AUC: {eval_dict["roc_auc"]:.4f}')
print(f'  Accuracy: {eval_dict["accuracy"]:.4f}')


## 5. Extract Probe Weights

Extract the learned weights from the trained probe for visualization


In [ ]:
# Extract probe weights
if probe_type == "mean":
    # For sklearn probe
    probe_weights = probe.model.coef_[0]  # Shape: (hidden_dim,)
    probe_bias = probe.model.intercept_[0]
    print(f"Probe weights shape: {probe_weights.shape}")
    print(f"Probe bias: {probe_bias:.4f}")
    
elif "torch" in probe_type:
    # For PyTorch probes
    if probe_type == "mean_torch":
        probe_weights = probe.model.linear.weight.detach().cpu().numpy()[0]  # Shape: (hidden_dim,)
        probe_bias = probe.model.linear.bias.detach().cpu().numpy()[0]
    elif probe_type == "attention_torch":
        # For attention probe, get the query/key/value weights
        probe_weights = {
            'W_Q': probe.model.W_Q.detach().cpu().numpy(),
            'W_K': probe.model.W_K.detach().cpu().numpy(),
            'W_V': probe.model.W_V.detach().cpu().numpy(),
            'W_out': probe.model.W_out.detach().cpu().numpy()
        }
        probe_bias = None
    print(f"Extracted probe weights from PyTorch model")

# Summary
print(f"\n✓ Probe weights extracted and ready for visualization")
print(f"\nSummary:")
print(f"  - Activations: {activations_tensor.shape}")
print(f"  - Questions/outputs: {len(dataset_df)} samples")
print(f"  - Probe performance: {eval_dict['roc_auc']:.4f} ROC-AUC")
print(f"  - Layer: {cfg.layer}")


## 6. Example: Aligning Tokens with Activations

For feature visualization, you'll need to tokenize questions and align them with activations


In [ ]:
# Example: How to align tokens with activations
# You need:
# 1. Your original questions (text) - available in dataset_df
# 2. The tokenizer used by your model
# 3. The saved activations (shape: [batch, seq_len, hidden_dim])

# Note: You'll need to load the appropriate tokenizer for your model
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

# For each question:
example_idx = 0
if 'input' in dataset_df.columns:
    question = dataset_df.iloc[example_idx]['input']
    print(f"Example question: {question[:200]}...")
    print(f"\nActivation for this sample: {activations_tensor[example_idx].shape}")
    print(f"Attention mask: {attention_mask[example_idx].sum().item():.0f} real tokens")
    
    # To tokenize and align:
    # tokens = tokenizer.tokenize(question)
    # token_ids = tokenizer.encode(question)
    # 
    # For each token, you can get the corresponding activation:
    # for i, (token, activation) in enumerate(zip(tokens, activations_tensor[example_idx])):
    #     probe_score = np.dot(activation.numpy(), probe_weights) + probe_bias
    #     print(f"Token {i}: '{token}' -> Probe score: {probe_score:.3f}")


In [ ]:
# You need three things:
# 1. Your original questions (text)
# 2. The tokenizer used by your model
# 3. The saved activations (shape: [batch, seq_len, hidden_dim] or similar)

# For each question:
question = "What is the capital of France?"
tokens = tokenizer.tokenize(question)  # or tokenizer.encode() depending on format

# If you have probe outputs already computed:
probe_activations = your_saved_probe_outputs  # shape: [seq_len] or [seq_len, num_probes]

# Align them:
for token, activation in zip(tokens, probe_activations):
    print(f"{token}: {activation}")